<div style="background: linear-gradient(135deg, #e9d8a6 0%, #ca6702 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">🗜️ 05 — PCA (Dimensionality Reduction)</h1>
  <p style="color: #d1f2eb; margin: 8px 0 0 0; font-size: 1.1em;">Fase 02 · Bloco 02 · Algoritmos</p>
</div>

## 🎯 Objetivos

- Usar PCA do sklearn como ferramenta prática de ML
- Visualizar separação de classes em dimensão reduzida
- Saber quando PCA ajuda e quando atrapalha

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

# Carregar MNIST reduzido (64 features)
digits = load_digits()
X, y = digits.data, digits.target
print(f'Shape original: {X.shape} → {X.shape[1]} features por amostra')

## 1️⃣ Reduzir 64D → 2D para Visualização

In [ ]:
# Normalizar primeiro!
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='tab10', s=10, alpha=0.6)
plt.colorbar(scatter, label='Dígito')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('MNIST 64D → 2D via PCA', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.show()

## 2️⃣ Quantos PCs Precisamos? Variância Acumulada

In [ ]:
pca_full = PCA().fit(X_scaled)
var_acum = np.cumsum(pca_full.explained_variance_ratio_) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(var_acum)+1), var_acum, 'b-o', markersize=3)
ax.axhline(95, color='r', linestyle='--', label='95%')
ax.axhline(90, color='orange', linestyle='--', label='90%')
n_95 = np.argmax(var_acum >= 95) + 1
ax.axvline(n_95, color='r', alpha=0.3)
ax.set_xlabel('Número de Componentes')
ax.set_ylabel('Variância Acumulada (%)')
ax.set_title(f'Scree Plot — {n_95} PCs para 95% da variância (de 64!)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 3️⃣ PCA + Classificador: Acelera Treino?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import time

# Sem PCA
t0 = time.time()
scores_full = cross_val_score(LogisticRegression(max_iter=5000), X_scaled, y, cv=5)
t_full = time.time() - t0

# Com PCA (mantendo 95% variância)
X_pca = PCA(n_components=n_95).fit_transform(X_scaled)
t0 = time.time()
scores_pca = cross_val_score(LogisticRegression(max_iter=5000), X_pca, y, cv=5)
t_pca = time.time() - t0

print(f'Sem PCA: {scores_full.mean():.4f} acc | {t_full:.2f}s | {X_scaled.shape[1]} features')
print(f'Com PCA: {scores_pca.mean():.4f} acc | {t_pca:.2f}s | {X_pca.shape[1]} features')
print(f'\n→ Speedup: {t_full/t_pca:.1f}x mais rápido com mínima perda de accuracy!')

## 🏋️ Questões

### Q1. Repita com `load_iris()`. Quantos PCs capturam 99% da variância?

### Q2. Compare SVM com e sem PCA em digits. PCA ajuda ou atrapalha?

### Q3. O que acontece se você NÃO normalizar antes de aplicar PCA?

In [ ]:
# Resolva aqui
